# Mini RAG — demo en vivo
**RAG = Retrieval Augmented Generation** (generación aumentada con recuperación)

Un LLM no sabe nada de **tus datos privados**: el manual de tu empresa, tus contratos, tus tickets. Y tampoco de lo que pasó después de su fecha de corte.

La solución RAG en 3 pasos:
1. **Retrieve**: buscar los fragmentos más relevantes para la pregunta.
2. **Augment**: meterlos en el prompt como contexto.
3. **Generate**: el LLM responde usando solo ese contexto, y cita de dónde lo saca.

En esta demo la «empresa» es **Bicis Nube**, una startup ficticia de alquiler de bicis eléctricas: el modelo no puede conocerla.

In [ ]:
!pip install -q --upgrade openai numpy

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook. Orden: secretos de Colab -> variable de entorno -> teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
print("Clave cargada:", os.environ["OPENAI_API_KEY"][:7] + "…")

La clave identifica tu cuenta y se factura contra ella. **Nunca** la escribas en el notebook: en Colab, guárdala en *Secretos* (icono de la llave) con el nombre `OPENAI_API_KEY`; en tu ordenador, como variable de entorno. Si no la encuentra, te la pide sin mostrarla.

In [ ]:
from openai import OpenAI

client = OpenAI()          # lee OPENAI_API_KEY del entorno
MODELO = "gpt-6-luna"      # el GPT-6 más rápido y barato (0,10 $ / 0,50 $ por millón de tokens)
ESFUERZO = "low"           # razonamiento: none | low | medium | high | xhigh | max

import numpy as np

def llm(pregunta, instrucciones=None, esfuerzo=ESFUERZO):
    r = client.responses.create(model=MODELO, instructions=instrucciones, input=pregunta,
                                reasoning={"effort": esfuerzo})
    return r.output_text.strip()

print(llm("Di hola en 5 palabras."))

## Paso 0 — El problema: el modelo no conoce tus datos
Preguntamos algo que solo sabe la empresa. Fíjate en qué hace el modelo: ¿reconoce que no lo sabe o se lo inventa?

In [ ]:
preguntas_imposibles = [
    "¿Cuánto cuesta el plan Plus de Bicis Nube?",
    "¿Qué pasa si devuelvo una bici de Bicis Nube con la batería por debajo del 10 %?",
]
for p in preguntas_imposibles:
    print(f"P: {p}\nR: {llm(p)[:250]}\n")

## Paso 1 — La base de conocimiento (documentos internos)

In [ ]:
documentos = [
    {"id": "tarifas-1", "texto": "Bicis Nube tiene tres planes: Básico (9 €/mes, 60 minutos al día), Plus (19 €/mes, minutos ilimitados) y Empresa (precio a medida, desde 10 bicis)."},
    {"id": "tarifas-2", "texto": "Sin plan, el alquiler cuesta 0,25 € por minuto más 1 € de desbloqueo. Los estudiantes con carné tienen un 30 % de descuento en el plan Plus."},
    {"id": "bateria-1", "texto": "Si una bici se devuelve con la batería por debajo del 10 %, se cobra una penalización de 3 €. Por encima del 50 % se regalan 10 minutos para el siguiente viaje."},
    {"id": "bateria-2", "texto": "Las bicis tienen 70 km de autonomía. La app avisa cuando quedan menos de 8 km y muestra la estación de carga más cercana."},
    {"id": "averias-1", "texto": "Para reportar una avería: botón 'Problema con la bici' en la app, foto obligatoria. El viaje se detiene y no se cobra desde ese momento."},
    {"id": "seguro-1", "texto": "Todos los viajes incluyen seguro de responsabilidad civil. El robo de la bici durante un viaje sin candado puesto tiene una franquicia de 150 €."},
    {"id": "empresa-1", "texto": "El plan Empresa incluye panel de control, facturación mensual única y mantenimiento en las oficinas del cliente cada 15 días."},
    {"id": "zonas-1", "texto": "El servicio funciona en Madrid, Valencia y Bilbao. Aparcar fuera de zona cuesta 5 € y la app bloquea el fin del viaje."},
]
print(len(documentos), "documentos")

## Paso 2 — Retrieve: búsqueda semántica con embeddings
Cada texto se convierte en un vector (**embedding**). Textos con significado parecido quedan cerca, aunque usen palabras distintas.
Indexamos una vez y, por cada pregunta, buscamos los `top_k` vectores más cercanos (**similitud coseno**).

In [ ]:
def embeddings(textos):
    r = client.embeddings.create(model="text-embedding-3-small", input=textos)
    return np.array([d.embedding for d in r.data])

indice = embeddings([d["texto"] for d in documentos])   # una sola llamada para todos
print("Índice:", indice.shape, "(documentos x dimensiones)")

def recuperar(pregunta, top_k=2, ver=True):
    q = embeddings([pregunta])[0]
    sims = indice @ q                       # vectores normalizados: producto escalar = coseno
    orden = np.argsort(-sims)
    if ver:
        print(f"Pregunta: {pregunta}")
        for pos, i in enumerate(orden):
            print(f"  {'✓' if pos < top_k else ' '} sim={sims[i]:.3f}  [{documentos[i]['id']}] {documentos[i]['texto'][:70]}…")
    return [documentos[i] for i in orden[:top_k]]

recuperar("¿Cuánto pago si me paso de los minutos del plan barato?");

## Paso 3 — Augment + Generate (con citas)
El truco de RAG: pegar los fragmentos recuperados en las instrucciones, pedir que responda **solo** con ellos
y que cite el `id` de cada uno. Si no está en el contexto, debe decir que no lo sabe.

In [ ]:
def preguntar_con_rag(pregunta, top_k=2, ver_prompt=False):
    docs = recuperar(pregunta, top_k=top_k, ver=False)
    contexto = "\n".join(f"[{d['id']}] {d['texto']}" for d in docs)
    instrucciones = (
        "Eres el asistente de atención al cliente de Bicis Nube. Responde SOLO con la información del CONTEXTO "
        "y cita entre corchetes el id del fragmento que usas, p. ej. [tarifas-1]. "
        "Si la respuesta no está en el contexto, di: 'No tengo esa información'.\n\n"
        f"CONTEXTO:\n{contexto}"
    )
    if ver_prompt:
        print("─" * 60, "\nLO QUE RECIBE EL MODELO\n[instructions]\n" + instrucciones + f"\n[input] {pregunta}\n" + "─" * 60)
    return llm(pregunta, instrucciones)

print(preguntar_con_rag("¿Qué pasa si devuelvo la bici con la batería al 5 %?", ver_prompt=True))

## Demo — sin RAG frente a con RAG

In [ ]:
for p in ["¿Cuánto cuesta el plan Plus de Bicis Nube?", "¿Bicis Nube tiene descuento para estudiantes?",
          "Se me ha roto la bici de Bicis Nube a mitad de viaje, ¿qué hago?", "¿Bicis Nube funciona en Sevilla?",
          "¿Cuál es el teléfono de atención al cliente de Bicis Nube?"]:
    print(f"\n{'─'*60}\nP: {p}")
    print(f"  SIN RAG: {llm(p)[:160]}")
    print(f"  CON RAG: {preguntar_con_rag(p)}")

Fíjate en las dos últimas. **Sevilla** no aparece en los documentos: si recupera `zonas-1` deducirá que solo hay Madrid, Valencia y Bilbao;
si no lo recupera, dirá que no tiene la información. El **teléfono** no está en ninguna parte. En ambos casos lo importante es que
**no se lo invente**: para eso sirve la instrucción «si no está en el contexto, dilo».

## Extra 1 — La búsqueda semántica entiende sinónimos
Ninguna de estas preguntas comparte palabras clave con el documento correcto, y aun así lo encuentra.

In [ ]:
for p in ["¿Me cobran algo por dejarla casi sin carga?",      # bateria-1, sin decir "batería" ni "10 %"
          "Me han robado la bici, ¿cuánto me toca pagar?",     # seguro-1
          "¿Puedo dejarla aparcada en un pueblo de las afueras?"]:  # zonas-1
    recuperar(p, top_k=1)
    print()

## Extra 2 — ¿Cuánto debe pensar el modelo en un RAG?
En RAG la respuesta ya está en el contexto: casi siempre basta con `effort="none"` o `"low"` (rápido y barato).
Para preguntas que mezclan varios fragmentos y hacen cuentas, pensar un poco más ayuda.

In [ ]:
import time
p = "Soy estudiante y uso la bici 2 horas al día. ¿Me sale mejor el plan Básico o el Plus, y cuánto pago al mes?"
docs = "\n".join(f"[{d['id']}] {d['texto']}" for d in documentos if d["id"].startswith("tarifas"))
inst = f"Responde solo con este contexto y cita los ids.\nCONTEXTO:\n{docs}"
for esfuerzo in ["none", "medium"]:
    print(f"── effort={esfuerzo}")
    t0 = time.time()
    r = llm(p, inst, esfuerzo)
    print(r, f"\n   [{time.time()-t0:.1f} s]\n")

**Qué hemos visto:** RAG = buscar + pegar en el prompt + responder citando. La calidad depende sobre todo de **recuperar bien**
(el Lab de RAG avanzado añade chunking, búsqueda híbrida, re-ranking y evaluación).